# How complete is the Policy Atlas evidence report?

**Question.** When Policy Atlas is given the same sources that a published POST or CAPE briefing used, how well does its report capture the briefing's most important facts?

**How the eval works** (full detail in [README.md](README.md)):

1. For each briefing, a LLM extracts the reference list and the key points, then breaks each key point into atomic facts. A human checks the lists (`ground_truth/<doc_id>/`).
2. Each kept reference is resolved to text (full text if possible, otherwise title + abstract) (`00_update_ground_truth.py`): OpenAlex metadata first, then a fetch of the reference's own URL. The outcome per reference (i.e. whether we could obtain at least abstract, or full text, or whether it could not be retrieved) is in `corpus_resolution.csv`.
3. The resolved references are ingested as the corpus of a Policy Atlas task, pre-screened as relevant, and the post-screening pipeline runs (`02_run_evals.py`). The Policy Atlas report is rendered as markdown.
4. A judge LLM grades every key fact against the report, one call per fact (`prompts/judge_v1.txt`).

**Data used here.** The run folder `results/runs/2026-11-05-deep-necessary-facts/`: per briefing:
* `report.md`: the Policy Atlas evidence report
* `facts.csv` (one row per fact with its verdict as to whether the PA report captured it or not)
* `corpus.csv` (one row per ingested reference)
* `eval_results.json` (scores and run metadata).
* Report citation counts come from the local dev database, cached to `report_citations.csv` in the run folder.

**Sections.**
1. Which briefings are in the test set, and their research questions
2. How many references each briefing has, and how many Policy Atlas could reach
3. The average number of references in a POST document
4. Briefing references vs citations in the Policy Atlas report
5. The coverage statistics, explained and recomputed
6. The distribution of verdicts
7. Final coverage metrics: best and worst briefings
8. Does report length or citation count predict coverage?
9. Does the judge need calibration?

## Findings

Run `2026-11-05-deep-necessary-facts`, 15 briefings, 668 key facts.

- **Coverage.** Pooled over all facts, 36% are fully represented and 73% are fully or partially represented. Per briefing, strict coverage runs from 0.00 (business rates rapid response) to 0.59 (ultra-processed foods); lenient from 0.55 to 0.94. Mean-of-briefings and pooled figures agree to within one point.
- **Verdict mix.** 38% of facts are partially represented: the report touches the topic but drops a figure or qualifier. 26% are not represented. Six facts (1%) are misrepresented; none are uncertain.
- **References.** A briefing in the run cites 131 references on average (median 154): POSTnotes about 190, POSTbriefs about 135, rapid responses about 40. Half of all references (975 of 1960) are not in OpenAlex; most of those are gov.uk and parliament pages fetched directly. A quarter (499) were never reached, mostly because the host blocked the fetch or the fetch failed. The Overton probe at the end of Section 2 answers how many of these Overton holds; on 2026-11-05 Overton rejected the key in `backend/.env` (HTTP 401), so its columns stay at zero until a valid `OVERTON_API_KEY` is set and the notebook is rerun.
- **Briefing vs report citations.** Reports carry 56 to 150 citation annotations but cite only 3 to 19 distinct sources, between 7% and 43% of the briefing's references. Stage-2 screening removes a large share of the ingested sources for some briefings (business rates 19 to 7, airports 32 to 18), and the report then cites 10% to 68% of what survives.
- **Length and citations do not predict coverage.** Spearman correlations between strict coverage and report length, report citations, briefing references or sources available are all between -0.22 and 0.12 over 15 briefings.
- **Judge.** Quotes are verbatim in the report for 489 of 498 verdicts that give one. Of the 9 misses, two quote the fact's own statement rather than the report. The six `misrepresented` verdicts include borderline calls, for example 98% vs "1.0 pregnancies per 100 woman-years" (99%) for Natural Cycles, and "associated with obesity" vs the report saying the evidence is not established, which reads as absent rather than contradicted. Seven facts were judged fully represented although none of their cited references reached synthesis; all are generic claims that other sources also state.

In [ ]:
from collections import Counter
import importlib
import json
import re
from urllib.parse import urlparse, urlsplit

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import Integer, cast, func, select

import _bootstrap  # noqa: F401
from policy_atlas.core.db import get_engine
from policy_atlas.core.prompt_fields import metadata_dict
from policy_atlas.core.schema import (
    annotation,
    artefact,
    block,
    chunk,
    citation,
    source_snapshot,
)
from utils import (
    DATASETS_DIR,
    GROUND_TRUTH_DIR,
    LABEL_SEP,
    RESULTS_DIR,
    ROOT,
    load_test_set,
)

# The scripts are numbered, so their names are not valid import statements.
_run_evals = importlib.import_module("02_run_evals")
VERDICTS, score = _run_evals.VERDICTS, _run_evals.score

RUN_DIR = RESULTS_DIR / "2026-11-05-deep-necessary-facts"
CITATIONS_CACHE = RUN_DIR / "report_citations.csv"
OVERTON_CACHE = DATASETS_DIR / "overton_probe.csv"
OVERTON_MAX_HITS = 50  # one Overton page; matching over 50 hits costs no extra requests
OVERTON_MATCHES = ["title", "url", "doi"]
OVERTON_RETRIES = 3  # per reference, on a network error with no HTTP status
OVERTON_RETRY_WAIT_S = 3
ENV_FILE = ROOT.parents[2] / "backend" / ".env"
OPENALEX_PREFIX = "https://openalex.org/"
# DATABASE_URL for the citation query, OVERTON_API_KEY for the probe. override=True so a
# key edited in backend/.env wins over the stale value a running kernel still holds.
load_dotenv(ENV_FILE, override=True)

# One fixed colour per category, never cycled: a blue ramp from "found" to "not found",
# grey for missing, red for a contradiction.
ROUTE_ORDER = [
    "in OpenAlex, full text",
    "in OpenAlex, abstract only",
    "not in OpenAlex, full text from URL",
    "in OpenAlex, no text or abstract",
    "not in OpenAlex, URL unreachable",
]
ROUTE_COLOURS = ["#2a78d6", "#9ec5f4", "#eb6834", "#898781", "#c3c2b7"]
VERDICT_COLOURS = {
    "fully_represented": "#2a78d6",
    "partially_represented": "#9ec5f4",
    "not_represented": "#c3c2b7",
    "misrepresented": "#e34948",
    "uncertain": "#eda100",
}
FUNNEL_COLS = [
    "briefing_references",
    "sources_ingested",
    "sources_after_stage2",
    "sources_cited_by_report",
]
FUNNEL_COLOURS = ["#0d366b", "#2a78d6", "#6da7ec", "#b7d3f6"]

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "axes.axisbelow": True,
    }
)

## 1. Which briefings are in the test set?

`inputs/POST_test_set.csv` lists 19 briefings. A briefing only enters this analysis when it has human-checked ground truth under `ground_truth/<doc_id>/` **and** a report in the run folder. The `Research question` column is the question the Policy Atlas task was seeded with (see `seed_task` in `02_run_evals.py`).

In [ ]:
test_set = pd.DataFrame(load_test_set())
DOC_IDS = sorted(p.name for p in RUN_DIR.iterdir() if p.is_dir())
records = {
    doc_id: json.loads((RUN_DIR / doc_id / "eval_results.json").read_text())
    for doc_id in DOC_IDS
}
briefing_type = test_set.set_index("Doc ID")["Type"]
test_set["in_this_run"] = test_set["Doc ID"].isin(DOC_IDS)
print(
    f"{len(test_set)} briefings in the test set; "
    f"{len(DOC_IDS)} have ground truth and a report in {RUN_DIR.name}"
)
test_set[["Doc ID", "Type", "Title", "Research question", "in_this_run"]]

In [ ]:
test_set.loc[~test_set["in_this_run"], ["Doc ID", "Type", "Title"]]

A range of briefing types is represented in our set of briefings:

In [ ]:
test_set["Type"].value_counts()

## 2. How many references does each briefing have, and how many could Policy Atlas reach?

**Where the numbers come from.** `references.csv` is the briefing's reference list (rows with `keep=y`). `corpus_resolution.csv` records what happened when `resolve_ref` in `00_update_ground_truth.py` tried to turn each kept reference into text:

- `full_text`: the production fetcher and parser got the document body.
- `abstract_only`: no full text, but OpenAlex had an abstract, so a one-chunk source was ingested.
- `dropped`: neither. The reference never reached Policy Atlas.

**OpenAlex.** `_openalex_work` looks the reference up by DOI, PubMed id or exact title. If it is found, `source_locator` is an `https://openalex.org/W...` id and the fetcher tries OpenAlex's candidate URLs before the reference's own URL. If it is not found, only the reference's own URL is tried.

**Overton.** The resolve step does **not** query Overton. Overton is a search backend in production, but this eval bypasses search entirely and builds the corpus from the briefing's own reference list. The group **not in OpenAlex** is mostly government, parliament and NGO web pages, the grey literature that Overton indexes; the host table below shows where they live. Whether Overton actually holds each reference is answered by a separate probe at the end of this section.

So each reference falls into one of five routes:

| route | meaning |
|---|---|
| in OpenAlex, full text | scholarly record, document fetched |
| in OpenAlex, abstract only | scholarly record, body paywalled or blocked, abstract used |
| not in OpenAlex, full text from URL | grey literature, fetched from its own URL |
| in OpenAlex, no text or abstract | scholarly record with nothing usable: **never reached** |
| not in OpenAlex, URL unreachable | grey literature that could not be fetched: **never reached** |

In [ ]:
def load_ground_truth(name: str) -> pd.DataFrame:
    """Stack ``ground_truth/<doc_id>/<name>.csv`` over the briefings in the run.

    Args:
        name: ``references``, ``key_facts`` or ``corpus_resolution``.

    Returns:
        Every row as strings, with a ``doc_id`` column.
    """
    return pd.concat(
        [
            pd.read_csv(
                GROUND_TRUTH_DIR / doc_id / f"{name}.csv",
                dtype=str,
                keep_default_na=False,
            ).assign(doc_id=doc_id)
            for doc_id in DOC_IDS
        ],
        ignore_index=True,
    )


def resolution_route(row: pd.Series) -> str:
    """Name the route by which one reference did, or did not, reach the corpus."""
    in_openalex = row["source_locator"].startswith(OPENALEX_PREFIX)
    if row["status"] == "dropped":
        return ROUTE_ORDER[3] if in_openalex else ROUTE_ORDER[4]
    if not in_openalex:
        return ROUTE_ORDER[2]
    return ROUTE_ORDER[0] if row["status"] == "full_text" else ROUTE_ORDER[1]


refs = load_ground_truth("references")
refs = refs[refs["keep"].str.strip().str.lower() == "y"]
resolution = load_ground_truth("corpus_resolution")
resolution["route"] = resolution.apply(resolution_route, axis=1)
briefing_references = refs.groupby("doc_id").size().rename("briefing_references")
# resolve runs over the kept references, so the two counts must agree.
assert briefing_references.equals(
    resolution.groupby("doc_id").size().rename("briefing_references")
)

routes = (
    resolution.pivot_table(
        index="doc_id", columns="route", values="label", aggfunc="size", fill_value=0
    )
    .reindex(columns=ROUTE_ORDER, fill_value=0)
    .rename_axis(columns=None)
)
routes.insert(0, "briefing_references", briefing_references)
routes["never_reached"] = routes[ROUTE_ORDER[3]] + routes[ROUTE_ORDER[4]]
routes["never_reached_share"] = (
    routes["never_reached"] / routes["briefing_references"]
).round(2)
# routes.loc["ALL"] = routes.sum()
# routes.loc["ALL", "never_reached_share"] = round(
#     routes.loc["ALL", "never_reached"] / routes.loc["ALL", "briefing_references"], 2
# )
routes.astype({c: int for c in routes.columns if c != "never_reached_share"})

The number of references in a briefing ranges from 27 to 231, with 131 being the mean.

In [ ]:
routes["briefing_references"].describe()

The histogram below suggests that there is not much of a pattern to how many references are in a briefing... 

In [ ]:
routes["briefing_references"].plot.hist(bins=15)

In [ ]:
routes.loc["ALL"] = routes.sum()

routes.loc["ALL", "never_reached_share"] = round(
    routes.loc["ALL", "never_reached"] / routes.loc["ALL", "briefing_references"], 2
)

routes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
routes.drop(index="ALL")[ROUTE_ORDER].plot.barh(
    stacked=False, ax=ax, color=ROUTE_COLOURS, width=0.7, edgecolor="white", linewidth=1
)
ax.invert_yaxis()
ax.set_xlabel("references in the briefing")
ax.set_ylabel("")
ax.set_title("How each briefing's references reached, or did not reach, the corpus")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

In [ ]:
# Why were references never reached? The last fetch attempt's error, per dropped reference.
dropped = resolution[resolution["status"] == "dropped"].copy()
dropped["last_error"] = dropped["attempts"].map(
    lambda a: a.split(" ; ")[-1].split(": ")[-1] if a else "no url to try"
)
dropped["last_error"].value_counts().head(10)

In [ ]:
# which hosts do the not-in-OpenAlex references live on? --> we might want to consider allowing PA to access some of these domains (if the source cannot be found in Overton)
not_in_openalex = resolution[
    ~resolution["source_locator"].str.startswith(OPENALEX_PREFIX)
]
hosts = not_in_openalex["source_locator"].map(
    lambda u: urlparse(u).netloc.removeprefix("www.") or "(no url)"
)
print(f"{len(not_in_openalex)} of {len(resolution)} references are not in OpenAlex")
(
    pd.crosstab(hosts, not_in_openalex["status"])
    .assign(total=lambda t: t.sum(axis=1))
    .sort_values("total", ascending=False)
    .head(15)
)

### Overton probe: how many references could Overton have supplied?

Production search has two backends: OpenAlex for scholarly records and Overton for policy documents. The resolve step used only OpenAlex, so this step asks the other half directly. For each kept reference it runs one Overton semantic search (`OvertonLiveBackend.search` in `search_live.py`) with the reference title and keeps the first page of 50 hits. A hit counts as the reference when its document URL equals the reference URL, or its DOI equals the reference DOI, or its title matches after normalisation (`norm_title` in `utils.py`). Overton has no URL or DOI lookup (those parameters are ignored), and its semantic search ranks similar documents above exact titles: on a random sample of 18 grey-literature references, one title search found 5 to 6 within 50 hits, and a document known to be in Overton ranked sixth. Treat the Overton counts as a lower bound. References with no title (a bare URL or DOI in the reference list) are not searched, because Overton has no URL or DOI lookup.

"Could in theory have been obtained" means Overton holds a record for the reference. Whether the pipeline's generated search queries would surface it is the search eval's question, not this one's.

Practicalities: Overton allows one request every 1.2 s, so the full set of references takes about 40 minutes. Results are cached per reference in `datasets/overton_probe.csv` and the probe resumes from the cache, so rerunning the notebook costs nothing once it has completed. It needs `OVERTON_API_KEY` in `backend/.env`. If the key is missing or rejected the probe stops and the table counts those references as `not probed`. Overton answers some long titles with an empty body; such a reference is recorded as `error` after three attempts and the probe carries on.

In [ ]:
OUTPUT_PATH = RUN_DIR / "overton_title_matches.jsonl"

results_df = pd.read_json(
    OUTPUT_PATH,
    lines=True,
)

In [ ]:
results_df.tail()

In [ ]:
results_df[results_df["total_results"] < 1000]["total_results"].hist(bins=100)

In [ ]:
results_df[results_df["total_results"] < 500]["total_results"].hist(bins=100)

In [ ]:
results_df[results_df["total_results"] > 0]["total_results"].hist(bins=100)

In [ ]:
results_df["total_results"].hist(bins=100)

In [ ]:
results_df["total_results"].describe()

In [ ]:
results_df = results_df.drop_duplicates(
    subset="input_title",
    keep="last",
)

flat_results = results_df.explode("candidates")

flat_results

In [ ]:
candidate_fields = pd.json_normalize(flat_results["candidates"].dropna())

In [ ]:
candidate_fields

In [ ]:
# Overton lookup is keyed by the stripped reference title, which is what
# determine_which_refs_are_in_overton.py searched. "Available" = at least one exact match.
with open(RUN_DIR / "overton_title_matches.jsonl") as f:
    in_overton = {
        r["input_title"]: any(c.get("url_match") for c in r["candidates"])
        for r in map(json.loads, f)
    }

in_overton

In [ ]:
availability = refs[["doc_id", "label", "title"]].merge(
    resolution[["doc_id", "label", "source_locator"]],
    on=["doc_id", "label"],
    validate="one_to_one",
)
availability["OpenAlex"] = availability["source_locator"].str.startswith(
    OPENALEX_PREFIX
)
# Five kept references have an empty title, so the probe never searched them: not in Overton.
availability["Overton"] = (
    availability["title"].str.strip().map(in_overton).fillna(False).astype(bool)
)

SOURCE_ORDER = ["both", "OpenAlex only", "Overton only", "neither"]
SOURCE_COLOURS = ["#2a78d6", "#9ec5f4", "#eb6834", "#c3c2b7"]
availability["source"] = pd.Series(SOURCE_ORDER)[
    (~availability["OpenAlex"]).astype(int) * 2 + (~availability["Overton"]).astype(int)
].to_numpy()

sources = pd.crosstab(availability["doc_id"], availability["source"]).reindex(
    columns=SOURCE_ORDER, fill_value=0
)
sources.loc["ALL"] = sources.sum()
sources

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sources.drop(index="ALL").plot.barh(
    stacked=True, ax=ax, color=SOURCE_COLOURS, width=0.7, edgecolor="white", linewidth=1
)
ax.invert_yaxis()
ax.set_xlabel("references in the briefing")
ax.set_ylabel("")
ax.set_title("Where each briefing's references could have come from")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

In [ ]:
sources

In [ ]:
source_locations = (
    availability[availability["source"] == "neither"]["source_locator"]
    .unique()
    .tolist()
)

In [ ]:
def get_most_common_domains(filtered_list, n=10):

    filtered_list = [s for s in filtered_list if s.startswith("https://")]

    domains = [f"{urlsplit(s).scheme}://{urlsplit(s).netloc}" for s in filtered_list]

    domain_counts = Counter(domains)

    # Get the 10 most common domains
    return domains, domain_counts, domain_counts.most_common(n)

In [ ]:
domains, domain_counts, most_common_domains = get_most_common_domains(source_locations)
most_common_domains

In [ ]:
# Find which domains are most common for Overton only sources
domains_o, domain_counts_o, most_common_domains_o = get_most_common_domains(
    availability[availability["source"] == "Overton only"]["source_locator"]
    .unique()
    .tolist()
)
most_common_domains_o

## 3. The average number of references in a POST document

Over the briefings in this run, with the test-set `Type` for a breakdown. The four briefings with no ground truth are not counted.

In [ ]:
per_briefing = briefing_references.to_frame().join(briefing_type.rename("Type"))
print(per_briefing["briefing_references"].describe().round(1))
per_briefing.groupby("Type")["briefing_references"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1)

## 4. Briefing references vs citations in the Policy Atlas report

The references pass through a funnel before the report can cite them:

1. **briefing_references**: kept rows of `references.csv`.
2. **sources_ingested**: references that resolved to full text or an abstract (`corpus.csv`).
3. **sources_after_stage2**: ingested sources that stage-2 screening (`screen_full`) did not mark `not_relevant`. Abstract-only sources have no full text to screen and are kept.
4. **sources_cited_by_report**: distinct sources with at least one citation in the report.

**report_citations** counts citation annotations: one per cited sentence of the report, so one source can be cited many times. `report.md` has its citation markers stripped (see `_report_markdown` in `runtime/inherit.py`), so the counts are read from the citation table in the dev database by task and run id, with the same joins as `match_up_citations.ipynb`, and cached to `report_citations.csv` in the run folder.

In [ ]:
def query_report_citations(engine, doc_id: str) -> list[dict]:
    """One row per source cited in a briefing's report, with citation counts.

    Args:
        engine: SQLAlchemy engine for the dev database that holds the run's tasks.
        doc_id: Briefing slug; its task and run ids come from ``eval_results.json``.
    """
    meta = records[doc_id]
    stmt = (
        select(
            source_snapshot.c.metadata,
            func.count().label("n_citations"),
            func.sum(cast(citation.c.verification_result == "pass", Integer)).label(
                "n_verified"
            ),
        )
        .select_from(artefact)
        .join(block, block.c.artefact_id == artefact.c.artefact_id)
        .join(annotation, annotation.c.block_id == block.c.block_id)
        .join(citation, citation.c.annotation_id == annotation.c.annotation_id)
        .join(chunk, chunk.c.chunk_id == citation.c.chunk_id)
        .join(
            source_snapshot,
            source_snapshot.c.source_snapshot_id == chunk.c.source_snapshot_id,
        )
        .where(
            artefact.c.task_id == meta["task_id"],
            artefact.c.capability_run_id == meta["capability_run_id"],
            annotation.c.annotation_type == "citation",
        )
        .group_by(source_snapshot.c.source_snapshot_id, source_snapshot.c.metadata)
    )
    with engine.connect() as conn:
        return [
            {
                "doc_id": doc_id,
                "ref_label": metadata_dict(metadata)["eval_ref_label"],
                "title": metadata_dict(metadata).get("title", ""),
                "n_citations": n_citations,
                "n_verified": n_verified,
            }
            for metadata, n_citations, n_verified in conn.execute(stmt)
        ]


def load_report_citations() -> pd.DataFrame:
    """Report citations per (briefing, source), from the cache or else the database."""
    if CITATIONS_CACHE.exists():
        return pd.read_csv(CITATIONS_CACHE, dtype={"ref_label": str})
    engine = get_engine()
    citations = pd.DataFrame(
        [row for doc_id in DOC_IDS for row in query_report_citations(engine, doc_id)]
    )
    citations.to_csv(CITATIONS_CACHE, index=False)
    return citations


citations = load_report_citations()
corpus = pd.concat(
    [
        pd.read_csv(
            RUN_DIR / doc_id / "corpus.csv", dtype=str, keep_default_na=False
        ).assign(doc_id=doc_id)
        for doc_id in DOC_IDS
    ],
    ignore_index=True,
)
funnel = pd.DataFrame(
    {
        "briefing_references": briefing_references,
        "sources_ingested": corpus.groupby("doc_id").size(),
        "sources_after_stage2": corpus[corpus["stage2_status"] != "not_relevant"]
        .groupby("doc_id")
        .size(),
        "sources_cited_by_report": citations.groupby("doc_id")["ref_label"].nunique(),
        "report_citations": citations.groupby("doc_id")["n_citations"].sum(),
    }
)
funnel["cited_share_of_briefing_refs"] = (
    funnel["sources_cited_by_report"] / funnel["briefing_references"]
).round(2)
funnel["cited_share_of_available"] = (
    funnel["sources_cited_by_report"] / funnel["sources_after_stage2"]
).round(2)
funnel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
funnel[FUNNEL_COLS].plot.barh(ax=ax, color=FUNNEL_COLOURS, width=0.8)
ax.invert_yaxis()
ax.set_xlabel("number of sources")
ax.set_ylabel("")
ax.set_title("From briefing reference list to sources the report cites")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

## 5. The coverage statistics, explained

Every kept fact gets exactly one verdict from the judge:

| verdict | the report ... |
|---|---|
| `fully_represented` | states the fact: same finding, same direction, same figures |
| `partially_represented` | covers the topic and agrees in direction, but omits a figure, qualifier or part of the claim |
| `not_represented` | does not cover the fact |
| `misrepresented` | contradicts the fact: different direction, materially different figure or opposite conclusion |
| `uncertain` | is too vague to tell |

From those, `score` in `02_run_evals.py` computes six coverage numbers per briefing. Each is a **share of facts**:

- **`coverage_all`** = facts judged `fully_represented` / all kept facts. The strict headline number.
- **`coverage_ingested`** = the same, but the denominator is only facts with at least one cited reference in the corpus (`ingested=y` in `facts.csv`). A fact whose every reference was dropped can never be covered, so the gap between `coverage_all` and `coverage_ingested` is the cost of unfetchable sources, not of the pipeline. It removes only part of that cost: a fact carries every reference of its key point, so it counts as ingested when one of them was retrieved, even if the reference that supports it was dropped. Read `coverage_ingested` as a lower bound on coverage of the facts the pipeline had access to. The cell below prints how many facts this excludes; it is a small number, so the two measures stay close.
- **`coverage_all_refs_ingested`** = the same, but the denominator is only facts whose key point has every cited reference in the corpus. The pipeline had the source of each of these facts, so a miss here is a synthesis miss. The subset is much smaller and is empty for some briefings (shown as `NaN`). Key points with few references are over-represented in it, so it is an estimate, not an upper bound.
- **`coverage_all_lenient`**, **`coverage_ingested_lenient`** and **`coverage_all_refs_ingested_lenient`**: the numerator also counts `partially_represented`. The gap between strict and lenient is the share of facts the report touches but states incompletely.

Two things to keep in mind when reading them:

- They are means over **facts**, so a key point split into five facts weighs five times one split into one. Section 7 also shows a per-point reweighting.
- A share over 15 briefings can be averaged two ways: the mean of the 15 per-briefing shares (each briefing counts equally) or the pooled share over all facts (each fact counts equally; long briefings dominate). Both are reported.

The cell below reloads `facts.csv` and `corpus.csv` and recomputes the scores with the same `score` function, as a check that `eval_results.json` and the CSVs agree. The `all_refs_ingested` columns are taken from that recomputation, because runs saved before the metric was added do not have them in `eval_results.json`.

In [ ]:
facts = pd.concat(
    [
        pd.read_csv(
            RUN_DIR / doc_id / "facts.csv", dtype=str, keep_default_na=False
        ).assign(doc_id=doc_id)
        for doc_id in DOC_IDS
    ],
    ignore_index=True,
)
scores = pd.DataFrame({doc_id: r["scores"] for doc_id, r in records.items()}).T
SCORE_COLS = [
    "coverage_all",
    "coverage_ingested",
    "coverage_all_lenient",
    "coverage_ingested_lenient",
]
ALL_REFS_COLS = [
    "n_all_refs_ingested",
    "coverage_all_refs_ingested",
    "coverage_all_refs_ingested_lenient",
]

for doc_id in DOC_IDS:
    rows = facts[facts["doc_id"] == doc_id].to_dict("records")
    recomputed = score(
        rows,
        {r["fact_id"]: r["verdict"] for r in rows},
        set(corpus.loc[corpus["doc_id"] == doc_id, "label"]),
    )
    for col in SCORE_COLS:
        assert abs(recomputed[col] - scores.loc[doc_id, col]) < 1e-9, (doc_id, col)
    for col in ALL_REFS_COLS:
        scores.loc[doc_id, col] = recomputed[col]
print("eval_results.json scores match a recomputation from facts.csv and corpus.csv")
print("facts with no ingested reference:", (facts["ingested"] != "y").sum())
scores[["n_facts", "n_ingested", *SCORE_COLS, *ALL_REFS_COLS]].round(2)

In [ ]:
# scores
fig, ax = plt.subplots(figsize=(10, 7))
scores[["coverage_ingested", "coverage_ingested_lenient"]].plot.barh(
    ax=ax, color=FUNNEL_COLOURS, width=0.8
)
ax.invert_yaxis()
ax.set_xlabel("number of sources")
ax.set_ylabel("")
ax.set_title("From briefing reference list to sources the report cites")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

## 6. The distribution of verdicts

First pooled over every fact in the run, then per briefing. Briefings are ordered by strict coverage.

In [ ]:
verdict_counts = (
    pd.crosstab(facts["doc_id"], facts["verdict"])
    .reindex(columns=VERDICTS, fill_value=0)
    .rename_axis(columns=None)
)
overall = verdict_counts.sum()
pd.DataFrame({"facts": overall, "share": (overall / overall.sum()).round(3)})

In [ ]:
verdict_share = verdict_counts.div(verdict_counts.sum(axis=1), axis=0)
ranked = scores.sort_values("coverage_all", ascending=False).index

fig, ax = plt.subplots(figsize=(10, 6))
verdict_share.loc[ranked].plot.barh(
    stacked=True,
    ax=ax,
    color=[VERDICT_COLOURS[v] for v in VERDICTS],
    width=0.7,
    edgecolor="white",
    linewidth=1,
)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("share of the briefing's key facts")
ax.set_ylabel("")
ax.set_title("Verdict mix per briefing (ordered by strict coverage)")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()
verdict_counts.loc[ranked].assign(n_facts=verdict_counts.sum(axis=1))

## 7. Final coverage metrics: which briefings did Policy Atlas do best and worst on?

In [ ]:
summary = (
    scores[["n_facts", *SCORE_COLS]]
    .join(briefing_type.rename("type"))
    .join(funnel)
    .join(
        pd.Series(
            {d: r["report_chars"] for d, r in records.items()}, name="report_chars"
        )
    )
)
# Per-point reweighting: mean over key points of the share of the point's facts fully represented.
facts["fully"] = facts["verdict"] == "fully_represented"
summary["coverage_per_point"] = (
    facts.groupby(["doc_id", "point_id"])["fully"].mean().groupby("doc_id").mean()
)
pooled = facts["fully"].mean()
pooled_lenient = (
    facts["verdict"].isin(["fully_represented", "partially_represented"]).mean()
)
print(
    f"mean of per-briefing coverage_all:          {summary['coverage_all'].mean():.3f}"
)
print(f"pooled coverage_all over all facts:         {pooled:.3f}")
print(
    f"mean of per-briefing coverage_all_lenient:  {summary['coverage_all_lenient'].mean():.3f}"
)
print(f"pooled coverage_all_lenient over all facts: {pooled_lenient:.3f}")
coverage_table = summary.sort_values("coverage_all", ascending=False)[
    ["type", "n_facts", "coverage_all", "coverage_all_lenient", "coverage_per_point"]
].round(2)
coverage_table

In [ ]:
print("best three (strict):  ", list(coverage_table.index[:3]))
print("worst three (strict): ", list(coverage_table.index[-3:]))
print(
    "best three (lenient): ",
    list(summary.sort_values("coverage_all_lenient", ascending=False).index[:3]),
)
print(
    "worst three (lenient):",
    list(summary.sort_values("coverage_all_lenient").index[:3]),
)

ordered = summary.sort_values("coverage_all")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    ordered.index,
    ordered["coverage_all_lenient"],
    color="#9ec5f4",
    label="lenient (fully + partially)",
)
ax.barh(
    ordered.index,
    ordered["coverage_all"],
    color="#2a78d6",
    label="strict (fully represented)",
)
for i, (strict, lenient) in enumerate(
    zip(ordered["coverage_all"], ordered["coverage_all_lenient"])
):
    ax.text(strict + 0.01, i, f"{strict:.2f}", va="center", fontsize=8, color="#0b0b0b")
    ax.text(
        lenient + 0.01, i, f"{lenient:.2f}", va="center", fontsize=8, color="#52514e"
    )
ax.set_xlim(0, 1.08)
ax.set_xlabel("coverage (share of key facts)")
ax.set_title("Coverage per briefing")
ax.legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

## 8. Does report length or citation count predict coverage?

The expectation: a longer report, or one that cites more sources, has more room to state the briefing's facts. The predictors below are the report's length in characters, the number of citations in it, the number of references in the briefing, and the number of sources that survived to synthesis. Spearman's rank correlation is used because the relationships need not be linear and the sample is 15 briefings, which is small: a single briefing can move a correlation a lot, so read the scatter plots, not only the coefficients.

In [ ]:
summary

In [ ]:
PREDICTORS = {
    "report_chars": "report length (characters)",
    "report_citations": "citations in the report",
    "briefing_references": "references in the briefing",
    "sources_after_stage2": "sources available to synthesis",
}
summary.corr(method="spearman", numeric_only=True).loc[
    ["coverage_ingested", "coverage_ingested_lenient"], list(PREDICTORS)
].round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (col, label) in zip(axes.flat, PREDICTORS.items()):
    ax.scatter(
        summary[col], summary["coverage_ingested"], color="#2a78d6", s=36, zorder=3
    )
    for doc_id, row in summary.iterrows():
        ax.annotate(
            doc_id,
            (row[col], row["coverage_ingested"]),
            fontsize=6.5,
            xytext=(3, 3),
            textcoords="offset points",
        )
    rho = summary[[col, "coverage_ingested"]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"{label}  (Spearman rho = {rho:.2f})", fontsize=10)
    ax.set_ylabel("coverage_ingested")
    ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

## 9. Does the judge need calibration?

The judge is a model, so its verdicts are evidence, not truth. Four cheap checks that need no second model:

1. **Is the supporting quote really in the report?** The prompt asks for the shortest quote from the report that supports the verdict. A quote that is not in the report means the judge paraphrased or invented it, and the verdict rests on nothing checkable. Matching is on lower-cased text with straight quotes and single spaces; the judge often joins fragments with `...`, adds `[...]` insertions or a trailing full stop, so the quote is split on ellipses, bracketed text is removed, and every fragment must appear in the report. A quote that is the fact's own statement is the clearest sign of a judge grading from the fact, not the report.
2. **`misrepresented` verdicts.** These are rare and serious, so each one is listed for a human to read against the report.
3. **`not_represented` with a quote.** The prompt says to return an empty quote for `not_represented`; a quote here means the judge found something relevant and then said the fact is absent.
4. **Facts the report could not have had a source for.** A fact whose every cited reference was dropped at resolution or marked `not_relevant` at stage 2 had no source in front of the synthesis step. A `fully_represented` verdict on such a fact means either the report stated it from other sources (plausible for general claims) or the judge was lenient. The list is worth a look.

In [ ]:
def normalise(text: str) -> str:
    """Lower case, straight quotes, single spaces, for substring matching of judge quotes."""
    text = (
        text.replace("\u2019", "'")
        .replace("\u2018", "'")
        .replace("\u201c", '"')
        .replace("\u201d", '"')
    )
    return re.sub(r"\s+", " ", text).strip().lower()


def quote_fragments(quote: str) -> list[str]:
    """The verbatim pieces of a judge quote: split on ellipses, brackets and outer punctuation removed."""
    text = re.sub(r"\[[^\]]*\]", " ", normalise(quote).strip("\"'"))
    return [
        f.strip(" .;,") for f in re.split(r"\.\.\.|\u2026", text) if f.strip(" .;,")
    ]


reports = {
    doc_id: normalise((RUN_DIR / doc_id / "report.md").read_text())
    for doc_id in DOC_IDS
}
facts["quote_in_report"] = [
    all(fragment in reports[doc_id] for fragment in quote_fragments(quote))
    if quote.strip()
    else None
    for quote, doc_id in zip(facts["quote"], facts["doc_id"])
]
facts["quote_is_fact"] = [
    normalise(quote).strip(" .") == normalise(statement).strip(" .")
    for quote, statement in zip(facts["quote"], facts["statement"])
]
print("quotes that are the fact statement itself:", int(facts["quote_is_fact"].sum()))
pd.crosstab(
    facts["verdict"],
    facts["quote_in_report"]
    .map({True: "quote found", False: "quote NOT found"})
    .fillna("no quote"),
)

In [ ]:
# Check 1: quotes the judge gave that are not in the report - hallucinations
facts.loc[
    facts["quote_in_report"].eq(False),
    ["doc_id", "fact_id", "verdict", "statement", "quote"],
]

In [ ]:
# Check 2: every misrepresented verdict, for a human read.
facts.loc[
    facts["verdict"] == "misrepresented",
    ["doc_id", "fact_id", "statement", "quote", "rationale"],
]

In [ ]:
# Check 3: not_represented verdicts that still carry a quote.
facts.loc[
    (facts["verdict"] == "not_represented") & (facts["quote"].str.strip() != ""),
    ["doc_id", "fact_id", "statement", "quote", "rationale"],
]

In [ ]:
# Check 4: facts whose every cited reference was unavailable to synthesis.
available = (
    corpus[corpus["stage2_status"] != "not_relevant"]
    .groupby("doc_id")["label"]
    .agg(set)
)
facts["n_refs_available"] = [
    len(set(labels.split(LABEL_SEP)) & available[doc_id])
    for labels, doc_id in zip(facts["ref_labels"], facts["doc_id"])
]
no_source = facts[facts["n_refs_available"] == 0]
print(
    f"{len(no_source)} of {len(facts)} facts had no cited reference available to synthesis"
)
pd.crosstab(no_source["doc_id"], no_source["verdict"]).reindex(
    columns=VERDICTS, fill_value=0
).rename_axis(columns=None)

In [ ]:
no_source.loc[
    no_source["verdict"] == "fully_represented",
    ["doc_id", "fact_id", "statement", "ref_labels", "quote"],
]